# 04c — Bogotá O3 daily maximum 8-hour running mean (MDA8)

The user's own paper (Casallas et al. 2023) uses the **daily maximum 8-hour running mean
(MDA8)** for O3 -- the standard WHO/EPA ozone health metric, following Sokhi et al.
(2021) -- not a plain daily mean. Checked directly: SISAIRE's published daily O3 value
(`Data/COLOMBIA/BG_O3_reporte_sisaire.csv`, used as the `o3` target so far) matches a
**plain arithmetic daily mean** almost exactly (median ratio ~1.00-1.02 against a daily
mean computed from the hourly data, across 4 stations checked), and is consistently
20-65% *lower* than the true MDA8 (median ratio 0.61-0.82) -- SISAIRE is not reporting
MDA8. This notebook computes a proper MDA8 from the same hourly source used for NO
(`~/Documents/Postdoc_MSCA-Bridge/USA_Ozone/DATA_ORIGINAL/O3.csv`, 2013-2022, 20 Bogotá
stations), replacing the O3 pipeline's target.

**MDA8 method** (standard EPA-style, as commonly implemented for Sokhi-et-al.-style
studies): for each station, take an 8-hour trailing rolling mean of the QC'd hourly
series (requiring >=6 of 8 hours present for that rolling value to count, the common
relaxed-EPA rule), then the daily maximum of those rolling means; a day's MDA8 is kept
only if >=18 of the 24 possible rolling-window values that day are valid (a completeness
requirement analogous to EPA's, preventing a single lucky/QC'd 8h stretch in an
otherwise-sparse day from producing a spuriously high "daily max").


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/path/to/project")
sys.path.insert(0, str(PROJECT_ROOT / "Scripts"))

import numpy as np
import pandas as pd

from common.qc import chauvenet_outlier_mask, completeness_by_station_year, select_stations_years

RAW_PATH = Path("/path/to/home/Documents/Postdoc_MSCA-Bridge/USA_Ozone/DATA_ORIGINAL/O3.csv")
OUT_DIR = PROJECT_ROOT / "Data" / "processed" / "colombia" / "bogota"

raw = pd.read_csv(RAW_PATH)
raw = raw.rename(columns={raw.columns[0]: "datetime"})
raw["datetime"] = pd.to_datetime(raw["datetime"], format="%d/%m/%Y %H:%M", errors="coerce")
raw = raw.dropna(subset=["datetime"])
print(raw.shape, raw["datetime"].min(), raw["datetime"].max())
station_cols = [c for c in raw.columns if c != "datetime"]
print(f"{len(station_cols)} stations:", station_cols)


(87648, 19) 2013-01-01 00:00:00 2022-12-31 23:00:00
18 stations: ['BOLIVIA', 'CARVAJAL', 'CENTRO DE ALTO RENDIMIENTO', 'CIUDAD BOLIVAR', 'FONTIBON', 'GUAYMARAL', 'JAZMIN', 'KENNEDY', 'LAS FERIAS', 'MIN AMBIENTE', 'PUENTE ARANDA', 'SAN CRISTOBAL', 'SUBA', 'TUNAL', 'USAQUEN', 'USME', 'COLINA', 'MOVIL FONTIBON']


## Melt to long format, physical range check, Chauvenet outlier flagging (per station, on hourly values)

In [2]:
long = raw.melt(id_vars="datetime", value_vars=station_cols, var_name="station", value_name="o3")
long["o3"] = pd.to_numeric(long["o3"], errors="coerce")
print("raw non-null hourly obs:", long["o3"].notna().sum())

O3_RANGE = (0.0, 500.0)  # ppb -- matches common/qc.py's PHYSICAL_RANGES convention
bad = (long["o3"] < O3_RANGE[0]) | (long["o3"] > O3_RANGE[1])
print(f"[range check] o3: {int(bad.sum())} values outside {O3_RANGE} -> NaN")
long.loc[bad, "o3"] = np.nan

mask = long.groupby("station")["o3"].transform(lambda s: chauvenet_outlier_mask(s))
print(f"[Chauvenet] o3: flagged {int(mask.sum())} hourly outliers")
long.loc[mask, "o3"] = np.nan
print("non-null hourly obs after QC:", long["o3"].notna().sum())


raw non-null hourly obs: 914743
[range check] o3: 0 values outside (0.0, 500.0) -> NaN
[Chauvenet] o3: flagged 1909 hourly outliers
non-null hourly obs after QC: 912834


## Per-station MDA8: 8h rolling mean (>=6/8 valid) -> daily max, requiring >=18/24 valid rolling windows/day

In [3]:
dup = long.duplicated(subset=["datetime", "station"], keep=False)
if dup.any():
    print(f"[dedupe] {int(dup.sum())} duplicate (datetime, station) rows -- averaging")
    long = long.groupby(["datetime", "station"], as_index=False)["o3"].mean()

wide = long.pivot(index="datetime", columns="station", values="o3").sort_index()

mda8_frames = []
for station in station_cols:
    s = wide[station]
    roll8 = s.rolling(window=8, min_periods=6).mean()
    n_valid_per_day = roll8.notna().resample("D").sum()
    daily_max = roll8.resample("D").max()
    daily_max = daily_max.where(n_valid_per_day >= 18)
    mda8_frames.append(daily_max.rename(station))

mda8_wide = pd.concat(mda8_frames, axis=1)
mda8_long = mda8_wide.reset_index().melt(id_vars="datetime", var_name="station", value_name="o3_mda8")
mda8_long = mda8_long.rename(columns={"datetime": "date"})
print(mda8_long.shape, "valid station-days:", mda8_long["o3_mda8"].notna().sum())


[dedupe] 36 duplicate (datetime, station) rows -- averaging


(65736, 3) valid station-days: 37095


## Completeness filtering (>=70% station-years, matching the rest of the pipeline), then network daily mean

In [4]:
completeness = completeness_by_station_year(mda8_long, value_col="o3_mda8")
MIN_COVERAGE = 0.70
good_station_years = select_stations_years(completeness, threshold=MIN_COVERAGE)
print(f"{len(good_station_years)} / {len(completeness)} station-years meet >={MIN_COVERAGE:.0%} completeness")

mda8_long["year"] = mda8_long["date"].dt.year
keep_keys = set(zip(good_station_years["station"], good_station_years["year"]))
mda8_long["keep"] = mda8_long.apply(lambda r: (r["station"], r["year"]) in keep_keys, axis=1)
qualified = mda8_long[mda8_long["keep"]]
print("qualifying station-days:", qualified.shape)

network = qualified.groupby("date")["o3_mda8"].mean().reset_index()
network = network.sort_values("date").reset_index(drop=True)
print(network.shape, network["date"].min(), network["date"].max())
print("non-null network-day count:", network["o3_mda8"].notna().sum())
network.head()


101 / 128 station-years meet >=70% completeness
qualifying station-days: (36885, 5)
(3652, 2) 2013-01-01 00:00:00 2022-12-31 00:00:00
non-null network-day count: 3650


,date,o3_mda8
0,2013-01-01,27.977083
1,2013-01-02,20.622024
2,2013-01-03,18.491667
3,2013-01-04,17.497500
4,2013-01-05,17.295833


## Sanity check against the plain-daily-mean network series already in use

`o3_mda8` should be systematically higher than the SISAIRE plain-daily-mean `o3` (same
direction as the single-station checks above), and still strongly correlated with it
(same underlying photochemistry, different aggregation window).


In [5]:
old_network = pd.read_parquet(OUT_DIR / "bogota_network_daily_clean.parquet")[["date", "o3"]]
cmp = network.merge(old_network, on="date", how="inner").dropna()
print("overlap days:", len(cmp))
print("corr(o3_mda8, o3_plain_mean):", cmp["o3_mda8"].corr(cmp["o3"]))
print("median ratio o3_plain_mean / o3_mda8:", (cmp["o3"] / cmp["o3_mda8"]).median())
print("mean o3_mda8:", cmp["o3_mda8"].mean(), " mean o3_plain_mean:", cmp["o3"].mean())


overlap days: 2919
corr(o3_mda8, o3_plain_mean): 0.7349116870587483
median ratio o3_plain_mean / o3_mda8: 0.6883472144490473
mean o3_mda8: 20.14878952892581  mean o3_plain_mean: 16.133697840500673


## Save

In [6]:
out_path = OUT_DIR / "bogota_o3_mda8_daily.parquet"
network.to_parquet(out_path, index=False)
print("Saved:", out_path, network.shape)


Saved: /path/to/project/Data/processed/colombia/bogota/bogota_o3_mda8_daily.parquet (3652, 2)
